## 🤝 6. Simple Multi-Agent Orchestration (Workflows)

**Goal**: Build workflows that orchestrate multiple agents


In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from utils.config import get_foundry_client, get_env
from agent_framework import Agent, WorkflowBuilder, AgentResponse, WorkflowViz
from agent_framework.orchestrations import SequentialBuilder
from agent_framework.orchestrations import ConcurrentBuilder
from typing import cast

client = get_foundry_client()
print("✅ Connected to Foundry")

c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


2026-10-26 11:39:21,770 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.
✅ Connected to Foundry


---
### Part A — Sequential Workflow (Writer → Reviewer) ✍️
A simple two-step pipeline: one agent writes, another reviews.

In [2]:
# ── A1. Define agents ────────────────────────────────────────────────────────
from agent_framework import Message

writer = client.as_agent(
    name="Writer",
    instructions=(
        "You are a concise copywriter. Write a single, punchy marketing "
        "paragraph based on the user's prompt. Be creative and engaging."
    ),
)

reviewer = client.as_agent(
    name="Reviewer",
    instructions=(
        "You are a content reviewer. Review the previous assistant message. "
        "Provide brief, actionable feedback: what's good and what to improve. "
        "Then provide a final polished version taking into consideration the feedback provided."
    ),
)

print("✍️ Writer -> 📝 Reviewer — agents created")

# 2) Build sequential workflow: writer -> reviewer
workflow = SequentialBuilder(participants=[writer, reviewer]).build()

# 3) Run and collect outputs
outputs: list[list[Message]] = []
async for event in workflow.run("Write a tagline for a budget-friendly eBike.", stream=True):
    if event.type == "output":
        outputs.append(cast(list[Message], event.data))

if outputs:
    print("===== Final Conversation =====")
    for i, msg in enumerate(outputs[-1], start=1):
        name = msg.author_name or ("assistant" if msg.role == "assistant" else "user")
        print(f"{'-' * 60}\n{i:02d} [{name}]\n{msg.text}")

✍️ Writer -> 📝 Reviewer — agents created
===== Final Conversation =====


TypeError: 'AgentResponseUpdate' object is not iterable

---
### Part B — Concurrent Workflow (ConcurrentBuilder - Researcher, Marketer, Legal) 📊
Build a more Concurrent fan-out/fan-in with default aggregator.

In [3]:
from typing import Any


researcher = Agent(
    client=client,
    instructions=(
        "You're an expert market and product researcher. Given a prompt, provide concise, factual insights,"
        " opportunities, and risks."
    ),
    name="researcher",
)

marketer = Agent(
    client=client,
    instructions=(
        "You're a creative marketing strategist. Craft compelling value propositions and target messaging"
        " aligned to the prompt."
    ),
    name="marketer",
)

legal = Agent(
    client=client,
    instructions=(
        "You're a cautious legal/compliance reviewer. Highlight constraints, disclaimers, and policy concerns"
        " based on the prompt."
    ),
    name="legal",
)

# 2) Build a concurrent workflow
# Participants are either Agents (type of SupportsAgentRun) or Executors
workflow = ConcurrentBuilder(participants=[researcher, marketer, legal]).build()

# 3) Run with a single prompt and pretty-print the final combined messages
events = await workflow.run("We are launching a new budget-friendly electric bike for urban commuters.")
outputs = events.get_outputs()

if outputs:
    print("===== Final Aggregated Conversation (messages) =====")
    for output in outputs:
        messages: list[Message] | Any = output
        for i, msg in enumerate(messages, start=1):
            name = msg.author_name if msg.author_name else "user"
            print(f"{'-' * 60}\n\n{i:02d} [{name}]:\n{msg.text}")

# Generate workflow visualization
# print("Generating workflow visualization...")
# viz = WorkflowViz(workflow)
# # Print out the mermaid string.
# print("Mermaid string: \n=======")
# print(viz.to_mermaid())
# print("=======")
# # Print out the DiGraph string with internal executors.
# print("DiGraph string: \n=======")
# print(viz.to_digraph(include_internal_executors=True))
# print("=======")


===== Final Aggregated Conversation (messages) =====


TypeError: 'AgentResponse' object is not iterable

---
### ✅ What you demonstrated
| Capability | Agent Framework Feature |
|---|---|
| Sequential pipelines | `SequentialBuilder(participants=[...]).build()` |
| Concurrent orchestration | `ConcurrentBuilder(participants=[...]).build()` |

**Next →** [07_orchestrations_advanced.ipynb](./07_orchestrations_advanced.ipynb) — Advanced Orchestrations.